# Experiment 4: Multi-Task Validation & Quantization (Optimized)

**Tasks:** Seg, Pose, PTQ, QAT, INT8 export

**Speedup:** Using `--pretrained` and `--compile` for faster training.

In [ ]:
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q && pip install tqdm timm -q
import torch, os, json, glob
from pathlib import Path
print(f'GPU: {torch.cuda.get_device_name(0)}')

FLAGS = "--pretrained --compile"

## Part A: Segmentation

In [ ]:
get_ipython().system(f'python scripts/train.py --task seg --variant quantized --data coco128.yaml --imgsz 416 --epochs 100 --seed 42 --name seg-q-416-seed42 {FLAGS}')
get_ipython().system(f'python scripts/train.py --task seg --variant standard --data coco128.yaml --imgsz 416 --epochs 100 --seed 42 --name seg-std-416-seed42 {FLAGS}')
print('✅ Segmentation complete!')

## Part B: Pose Estimation

In [ ]:
get_ipython().system(f'python scripts/train.py --task pose --variant quantized --data coco8-pose.yaml --imgsz 416 --epochs 100 --seed 42 --name pose-q-416-seed42 {FLAGS}')
get_ipython().system(f'python scripts/train.py --task pose --variant standard --data coco8-pose.yaml --imgsz 416 --epochs 100 --seed 42 --name pose-std-416-seed42 {FLAGS}')
print('✅ Pose complete!')

## Part C: Quantization

In [ ]:
weights = 'experiments/results/ablation-a4-relu6/best.pt'
if not Path(weights).exists():
    print('Training quick baseline...')
    get_ipython().system(f'python scripts/train.py --task det --variant quantized --imgsz 416 --epochs 30 --seed 42 --name quant-baseline {FLAGS}')
    weights = 'experiments/results/quant-baseline/best.pt'
print(f'Using: {weights}')
get_ipython().system(f'python scripts/quantize.py --mode ptq --weights {weights} --task det --variant quantized --imgsz 416 --n-calib 500 --backend qnnpack')
print('✅ PTQ complete!')

In [ ]:
get_ipython().system(f'python scripts/quantize.py --mode qat --weights {weights} --task det --variant quantized --imgsz 416 --epochs 10 --lr 1e-4 --backend qnnpack')
print('✅ QAT complete!')